In [ ]:
import os
from pathlib import Path
import sys
from urllib.request import urlretrieve

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.pyplot as plt
import numpy as np
import scipy.io as sio
from rich.console import Console
from rich.table import Table

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "tdmd").exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from tdmd import DCTTransform, TDMDII

plt.style.use("ggplot")

DATA_URL = "https://raw.githubusercontent.com/dynamicslab/databook_python/master/DATA/VORTALL.mat"
GRID_SHAPE = (199, 449)
SNAPSHOT_COUNT = 150


In [ ]:
data_path = PROJECT_ROOT / "examples" / "data" / "VORTALL.mat"
data_path.parent.mkdir(parents=True, exist_ok=True)
if not data_path.exists():
    urlretrieve(DATA_URL, data_path)

data_path


In [ ]:
states = sio.loadmat(data_path)["VORTALL"][:, :SNAPSHOT_COUNT]
tensor = states.reshape(*GRID_SHAPE, SNAPSHOT_COUNT, order="F").transpose(0, 2, 1)
snapshot_idx = 50


In [ ]:
gamma = 0.99999
model = TDMDII(DCTTransform(), gamma=gamma)
model.fit(tensor)

reconstructed_tensor = np.asarray(model.reconstructed_data).real
reconstructed_states = reconstructed_tensor.transpose(0, 2, 1).reshape(states.shape, order="F")
state_errors = np.array([
    np.linalg.norm(states[:, idx] - reconstructed_states[:, idx]) / np.linalg.norm(states[:, idx])
    for idx in range(states.shape[1])
])

snapshot_true = states[:, snapshot_idx].reshape(GRID_SHAPE, order="F")
snapshot_pred = reconstructed_tensor[:, snapshot_idx, :]
snapshot_re = float(state_errors[snapshot_idx])

table = Table(title="Cylinder Flow TDMDII")
table.add_column("Name", no_wrap=True)
table.add_column("Value")
table.add_row("state matrix shape", f"{tuple(states.shape)}")
table.add_row("k_max", f"{int(model.multirank.max())}")
table.add_row("sum k_j", f"{int(model.multirank.sum())}")
table.add_row("mean state-wise RE", f"{state_errors.mean():.4e}")
table.add_row("snapshot RE", f"{snapshot_re:.4e}")
Console().print(table)


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4.8), constrained_layout=True)
axes[0].imshow(snapshot_true, origin="lower", cmap="viridis", aspect="auto")
axes[0].set_title(f"Original {snapshot_idx}")
axes[0].set_xticks([])
axes[0].set_yticks([])

axes[1].imshow(snapshot_pred, origin="lower", cmap="viridis", aspect="auto")
axes[1].set_title("TDMDII Reconstruction")
axes[1].set_xticks([])
axes[1].set_yticks([])

residual_img = axes[2].imshow(snapshot_true - snapshot_pred, origin="lower", cmap="coolwarm", aspect="auto")
axes[2].set_title("Residual")
axes[2].set_xticks([])
axes[2].set_yticks([])
fig.colorbar(residual_img, ax=axes[2], shrink=0.9, pad=0.02)

axes[3].plot(state_errors, linewidth=2, color="tab:green")
axes[3].axvline(snapshot_idx, color="black", linestyle="--", linewidth=1)
axes[3].set_title("State-wise RE")
axes[3].set_xlabel("Snapshot")
axes[3].set_ylabel("RE")
axes[3].grid(True, alpha=0.3)

plt.show()
